# Late Order Recovery: Agentic Supply-Chain Workflows on the NVIDIA Stack

**Workshop notebook** -- a single, end-to-end example of a model-driven agent
recovering a late customer shipment using structured tool calls, explicit skills,
and sequence-sensitive evaluation.

---
## 1. Introduction

### What this notebook demonstrates

This notebook walks through a **multi-step agentic workflow** for late-order
recovery in a supply-chain setting. Rather than building a general-purpose agent
platform, we focus on one rich scenario and show concrete patterns for:

- **Structured tool calling** using a Nemotron-style JSON schema
- **Explicit higher-level skills** composed from deterministic tools
- **Fallback parsing** for malformed or partially structured model outputs
- **Sequence-sensitive evaluation** that checks tool-call ordering, not just outcomes
- **Training-oriented patterns** aligned with `NeMo RL` DatumSpec exports and reward decomposition

### Why supply chain?

Supply-chain operations are a strong domain for agentic workflows because:

1. **Decisions depend on ordered information gathering.** You cannot score a
   transfer option without first knowing which alternate DC has stock, and you
   cannot check alternate stock without first confirming the primary source is
   short. This creates natural, machine-checkable sequence dependencies.

2. **Tools are deterministic but composition is not.** Each lookup (inventory,
   ETA, capacity) is a pure function, but the agent must decide *which* tools
   to call and in *what order* based on intermediate results.

3. **Mitigations involve tradeoffs.** Expediting from a supplier is faster but
   costlier; transferring from another DC is cheaper but slower. The agent
   must gather enough data to compare options before recommending.

### Why sequence correctness matters

Most agent evaluations check whether the final answer is correct. But in
supply-chain recovery, *how* the agent arrived at the answer matters:

- Calling `get_transfer_eta` before `find_alternate_inventory` is logically
  invalid -- you cannot estimate a transfer without knowing the source.
- Calling `score_recovery_options` before gathering any options is pointless.
- An agent that guesses the right answer without proper investigation would
  be unsafe in production.

We define and enforce these dependencies explicitly, making the evaluation
**sequence-sensitive** rather than outcome-only.

---
## 2. Scenario Definition

### Business problem

Customer order **SO-10482** for **1,200 units** of **SKU-4090** is at risk of
missing its committed delivery date. The order was placed against distribution
center **DC-WEST-01**, which is the primary source. Shipment tracking shows
the order has not yet shipped and the committed date is approaching.

### Target task

> Determine whether order `SO-10482` can still be fulfilled on time from its
> primary source. If not, recommend the best mitigation action from the
> available options: fulfill from original DC, transfer from an alternate DC,
> expedite from a supplier, partially fulfill, or substitute with a compatible
> SKU.

### Success criteria

A successful agent run must:

1. **Diagnose the risk** -- confirm the order exists, retrieve shipment status,
   and identify that the order is at risk.
2. **Check primary fulfillment** -- verify inventory and capacity at DC-WEST-01.
3. **Explore alternates** -- check at least one alternate DC or supplier path.
4. **Score and recommend** -- compare candidate options and produce a final
   recommendation with rationale.
5. **Respect sequence dependencies** -- never call a downstream tool before its
   prerequisites.
6. **Complete in 5-10 tool calls** -- efficient but thorough.

### Sequence dependencies (machine-checkable)

```
get_order
  -> get_shipment_status
  -> get_inventory(source DC)
  -> get_fulfillment_capacity(source DC)
  -> get_supplier_expedite_options

get_inventory(source DC)
  -> find_alternate_inventory
     -> get_transfer_eta

at least one mitigation path explored
  -> score_recovery_options
     -> recommend_action
```

### Mitigation options the agent should evaluate

| Option | Description |
|--------|-------------|
| Primary fulfill | Ship from DC-WEST-01 if enough stock and capacity |
| DC transfer | Transfer from an alternate DC with available inventory |
| Supplier expedite | Rush order directly from a supplier |
| Partial fulfillment | Ship what is available, backorder the rest |
| SKU substitute | Use a compatible alternate SKU if enabled |

---
## 3. Synthetic Data Model

All scenario data lives in `src/scenario_data.py` as plain Python dicts.
The data is small enough to print in full and inspect cell-by-cell.

Tables:
- **ORDERS** -- customer orders (keyed by order_id)
- **SHIPMENTS** -- shipment status per order
- **INVENTORY** -- on-hand inventory by (sku, dc_id)
- **TRANSFER_LANES** -- routes between DCs with lead times
- **SUPPLIER_OPTIONS** -- supplier expedite options by sku
- **FULFILLMENT_CAPACITY** -- available capacity by (dc_id, date)
- **SUBSTITUTE_SKUS** -- optional substitute SKU mappings

In [1]:
import sys, pathlib
# Ensure src/ is importable from the notebook
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
# Also allow running from repo root
sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [2]:
from src.scenario_data import (
    ORDERS, SHIPMENTS, INVENTORY, TRANSFER_LANES,
    SUPPLIER_OPTIONS, FULFILLMENT_CAPACITY, SUBSTITUTE_SKUS,
)

print("=== Orders ===")
for oid, o in ORDERS.items():
    print(f"  {oid}: {o['qty']} x {o['sku']} -> {o['source_dc']} (due {o['committed_date']})")

print("\n=== Shipments ===")
for oid, s in SHIPMENTS.items():
    print(f"  {oid}: status={s['status']}, shipped={s['shipped_qty']}")

print("\n=== Inventory ===")
for (sku, dc), inv in INVENTORY.items():
    print(f"  {sku} @ {dc}: on_hand={inv['on_hand']}, reserved={inv['reserved']}, available={inv['available']}")

print("\n=== Transfer Lanes ===")
for (fdc, tdc), lane in TRANSFER_LANES.items():
    print(f"  {fdc} -> {tdc}: {lane['lead_days']} days, ${lane['cost_per_unit']}/unit")

print("\n=== Supplier Expedite Options ===")
for sku, opts in SUPPLIER_OPTIONS.items():
    for opt in opts:
        print(f"  {sku} via {opt['supplier']}: {opt['lead_days']} days, ${opt['cost_per_unit']}/unit (MOQ {opt['moq']})")

print("\n=== Fulfillment Capacity ===")
for (dc, dt), cap in FULFILLMENT_CAPACITY.items():
    print(f"  {dc} on {dt}: {cap['remaining']}/{cap['max_units']} remaining")

print("\n=== Substitute SKUs ===")
for sku, subs in SUBSTITUTE_SKUS.items():
    for s in subs:
        print(f"  {sku} -> {s['substitute_sku']} ({s['compatibility']}): {s['notes']}")

=== Orders ===
  SO-10482: 1200 x SKU-4090 -> DC-WEST-01 (due 2026-04-18)
  SO-10483: 500 x SKU-100 -> DC-EAST-02 (due 2026-04-21)
  SO-10484: 2000 x SKU-200 -> DC-CENTRAL-03 (due 2026-04-15)
  SO-10485: 200 x SKU-300 -> DC-WEST-01 (due 2026-04-22)
  SO-10486: 600 x SKU-400 -> DC-EAST-02 (due 2026-04-19)
  SO-10487: 3000 x SKU-500 -> DC-CENTRAL-03 (due 2026-04-14)
  SO-10488: 800 x SKU-600 -> DC-EAST-02 (due 2026-04-22)
  SO-10489: 350 x SKU-700 -> DC-WEST-01 (due 2026-04-19)
  SO-10490: 100 x SKU-800 -> DC-CENTRAL-03 (due 2026-04-25)
  SO-10491: 1500 x SKU-900 -> DC-WEST-01 (due 2026-04-24)

=== Shipments ===
  SO-10482: status=pending, shipped=0
  SO-10483: status=pending, shipped=0
  SO-10484: status=pending, shipped=0
  SO-10485: status=in_progress, shipped=0
  SO-10486: status=pending, shipped=0
  SO-10487: status=pending, shipped=0
  SO-10488: status=pending, shipped=0
  SO-10489: status=pending, shipped=0
  SO-10490: status=pending, shipped=0
  SO-10491: status=pending, shipped=

---
## 4. Skill Definitions

Skills are **higher-level behaviors** composed from deterministic tools.
The agent selects a skill; the skill defines which tools to call and in what
order. This separation makes it possible to evaluate *skill selection* and
*tool use* independently.

| Skill | Purpose | Tools used |
|-------|---------|------------|
| `diagnose_order_risk` | Understand current order/shipment state | `get_order`, `get_shipment_status` |
| `assess_primary_fulfillment` | Check if source DC can still fulfill | `get_inventory`, `get_fulfillment_capacity` |
| `evaluate_alternate_recovery_paths` | Explore alternate DCs and suppliers | `find_alternate_inventory`, `get_transfer_eta`, `get_supplier_expedite_options` |
| `synthesize_recommendation` | Score options and recommend action | `score_recovery_options`, `recommend_action` |

### Skills vs. tools

- A **tool** is a single deterministic lookup or computation.
- A **skill** is a reusable behavior pattern that orchestrates multiple tools.
- The agent reasons at the **skill level** ("I should diagnose the order risk
  first") and the execution engine translates that into **tool-level** calls.

This two-layer design lets us:
1. Evaluate whether the agent picked the right skill for the situation.
2. Evaluate whether the tools within each skill were called correctly.
3. Supervise at the skill level for training (coarser, more robust signal).

### Skill transitions

Skills follow a strict diagnostic flow. Each skill has a defined set of
valid successors, enforced by `validate_skill_transition()`:

```
diagnose_order_risk -> assess_primary_fulfillment -> evaluate_alternate_recovery_paths -> synthesize_recommendation
```

This ordering is machine-checkable and feeds directly into the
**sequence correctness** evaluator.

In [3]:
from src.runtime.workflows import (
    SKILL_NAMES, SKILL_TOOL_PATTERNS, SKILL_TRANSITIONS, SKILL_ORDER,
    SKILL_REGISTRY, validate_skill_transition,
)

print("=== Skill catalog ===\n")
for skill in SKILL_NAMES:
    tools = SKILL_TOOL_PATTERNS[skill]
    print(f"{skill}:")
    print(f"  tools: {' -> '.join(tools)}")
    transitions = SKILL_TRANSITIONS.get(skill, set())
    print(f"  next:  {transitions if transitions else '(terminal)'}")
    print()

print("=== Transition validation examples ===\n")
# Valid
ok, reason = validate_skill_transition(None, "diagnose_order_risk")
print(f"  (start) -> diagnose_order_risk: valid={ok}")
# Invalid: skipping ahead
ok, reason = validate_skill_transition("diagnose_order_risk", "synthesize_recommendation")
print(f"  diagnose -> synthesize (skip): valid={ok}  ({reason})")

=== Skill catalog ===

diagnose_order_risk:
  tools: get_order -> get_shipment_status
  next:  {'assess_primary_fulfillment'}

assess_primary_fulfillment:
  tools: get_inventory -> get_fulfillment_capacity
  next:  {'evaluate_alternate_recovery_paths'}

evaluate_alternate_recovery_paths:
  tools: find_alternate_inventory -> get_transfer_eta -> get_supplier_expedite_options
  next:  {'synthesize_recommendation'}

synthesize_recommendation:
  tools: score_recovery_options -> recommend_action
  next:  (terminal)

=== Transition validation examples ===

  (start) -> diagnose_order_risk: valid=True
  diagnose -> synthesize (skip): valid=False  (Invalid transition: diagnose_order_risk -> synthesize_recommendation. Allowed: {'assess_primary_fulfillment'}.)


### Running skills step by step

Each skill takes a `SkillContext` that accumulates state across the
diagnostic flow. The context records every tool call made, making the
execution fully inspectable.

In [4]:
from src.runtime.workflows import (
    WorkflowContext, diagnose_order_risk, assess_primary_fulfillment,
    evaluate_alternate_recovery_paths, synthesize_recommendation,
)

# Create a context for the diagnostic flow
ctx = WorkflowContext(order_id="SO-10482")

# Skill 1: Diagnose the order risk
diagnosis = diagnose_order_risk(ctx)
print("=== Skill 1: diagnose_order_risk ===")
print(f"  at_risk:        {diagnosis.is_at_risk}")
print(f"  reason:         {diagnosis.reason}")
print(f"  days_remaining: {diagnosis.days_remaining}")
print(f"  sku:            {diagnosis.sku}")
print(f"  qty:            {diagnosis.qty}")
print(f"  source_dc:      {diagnosis.source_dc}")
print(f"  tool calls so far: {len(ctx.tool_calls)}")
print()

# Skill 2: Assess primary fulfillment
primary = assess_primary_fulfillment(ctx)
print("=== Skill 2: assess_primary_fulfillment ===")
print(f"  can_fulfill:   {primary.can_fulfill}")
print(f"  available_qty: {primary.available_qty}")
print(f"  shortfall:     {primary.shortfall}")
print(f"  capacity_ok:   {primary.capacity_ok}")
print(f"  tool calls so far: {len(ctx.tool_calls)}")
print()

# Skill 3: Evaluate alternate recovery paths
paths = evaluate_alternate_recovery_paths(ctx)
print("=== Skill 3: evaluate_alternate_recovery_paths ===")
for p in paths:
    flag = "OK" if p.feasible else "NOT FEASIBLE"
    print(f"  [{flag}] {p.path_type} from {p.source}: "
          f"eta={p.eta_days}d, ${p.cost_per_unit}/unit, avail={p.available_qty}")
print(f"  tool calls so far: {len(ctx.tool_calls)}")
print()

# Skill 4: Synthesize recommendation
rec = synthesize_recommendation(ctx)
print("=== Skill 4: synthesize_recommendation ===")
print(f"  action:              {rec.action}")
print(f"  expected_delivery:   {rec.expected_delivery}")
print(f"  meets_committed:     {rec.meets_committed_date}")
print(f"  confidence:          {rec.confidence}")
print(f"  rationale:           {rec.rationale}")
print(f"  total tool calls:    {len(ctx.tool_calls)}")

=== Skill 1: diagnose_order_risk ===
  at_risk:        True
  reason:         Order not yet shipped. 8 days until committed date.
  days_remaining: 8
  sku:            SKU-4090
  qty:            1200
  source_dc:      DC-WEST-01
  tool calls so far: 2

=== Skill 2: assess_primary_fulfillment ===
  can_fulfill:   False
  available_qty: 300
  shortfall:     900
  capacity_ok:   True
  tool calls so far: 4

=== Skill 3: evaluate_alternate_recovery_paths ===
  [OK] dc_transfer from DC-EAST-02: eta=4d, $2.5/unit, avail=1000
  [OK] supplier_expedite from supplier:GlobalChip Express: eta=7d, $8.0/unit, avail=5000
  [OK] supplier_expedite from supplier:FastSemi Direct: eta=5d, $12.0/unit, avail=1500
  [NOT FEASIBLE] substitute from substitute:SKU-4090-B@DC-WEST-01: eta=0d, $0.0/unit, avail=800
  tool calls so far: 8

=== Skill 4: synthesize_recommendation ===
  action:              dc_transfer from DC-EAST-02
  expected_delivery:   2026-04-14
  meets_committed:     True
  confidence:          

In [5]:
# Inspect the full tool call trace recorded in the context
print(f"=== Full tool call trace ({len(ctx.tool_calls)} calls) ===\n")
print(f"Skills executed: {' -> '.join(ctx.skills_executed)}\n")

for i, tc in enumerate(ctx.tool_calls, 1):
    print(f"  {i:2d}. [{tc.skill_name}] {tc.tool_name}({tc.arguments})")

print(f"\n  Total: {len(ctx.tool_calls)} tool calls across {len(ctx.skills_executed)} skills")

=== Full tool call trace (10 calls) ===

Skills executed: diagnose_order_risk -> assess_primary_fulfillment -> evaluate_alternate_recovery_paths -> synthesize_recommendation

   1. [diagnose_order_risk] get_order({'order_id': 'SO-10482'})
   2. [diagnose_order_risk] get_shipment_status({'order_id': 'SO-10482'})
   3. [assess_primary_fulfillment] get_inventory({'sku': 'SKU-4090', 'dc_id': 'DC-WEST-01'})
   4. [assess_primary_fulfillment] get_fulfillment_capacity({'dc_id': 'DC-WEST-01', 'date': '2026-04-18'})
   5. [evaluate_alternate_recovery_paths] find_alternate_inventory({'sku': 'SKU-4090', 'region': 'ALL'})
   6. [evaluate_alternate_recovery_paths] get_transfer_eta({'from_dc': 'DC-EAST-02', 'to_dc': 'DC-WEST-01', 'sku': 'SKU-4090', 'qty': 900})
   7. [evaluate_alternate_recovery_paths] get_transfer_eta({'from_dc': 'DC-CENTRAL-03', 'to_dc': 'DC-WEST-01', 'sku': 'SKU-4090', 'qty': 900})
   8. [evaluate_alternate_recovery_paths] get_supplier_expedite_options({'sku': 'SKU-4090', 'qty': 

---
## 5. Tool Definitions

Each tool is a pure function over the synthetic data tables. Tools are
registered in `TOOL_REGISTRY` so the agent loop can dispatch by name.

| Tool | Inputs | Output | Dependencies |
|------|--------|--------|-------------|
| `get_order` | order_id | Order details | (none) |
| `get_shipment_status` | order_id | Shipment state | get_order |
| `get_inventory` | sku, dc_id | Stock levels | get_order |
| `find_alternate_inventory` | sku, region | Alternate DC stock | get_inventory |
| `get_transfer_eta` | from_dc, to_dc, sku, qty | Transfer estimate | find_alternate_inventory |
| `get_supplier_expedite_options` | sku, qty | Supplier rush options | get_order |
| `get_fulfillment_capacity` | dc_id, date | DC capacity | get_order |
| `score_recovery_options` | options, objective | Ranked options | >= 1 mitigation path |
| `recommend_action` | context | Final recommendation | score_recovery_options |

In [6]:
from src.runtime.tools import TOOL_REGISTRY, TOOL_DEPENDENCIES

print(f"Registered tools: {len(TOOL_REGISTRY)}\n")

print("Tool catalog:")
for name, (fn, params, desc) in TOOL_REGISTRY.items():
    plist = ", ".join(f"{k}: {v}" for k, v in params.items())
    print(f"  {name}({plist})")
    print(f"    {desc}")
    deps = TOOL_DEPENDENCIES.get(name, set())
    print(f"    depends on: {deps if deps else '(none)'}")
    print()

print("--- Example tool calls ---\n")

import json

# 1. Look up the order
result = TOOL_REGISTRY["get_order"][0](order_id="SO-10482")
print("get_order('SO-10482'):")
print(json.dumps(result, indent=2))

# 2. Check primary DC inventory
result = TOOL_REGISTRY["get_inventory"][0](sku="SKU-4090", dc_id="DC-WEST-01")
print(f"\nget_inventory('SKU-4090', 'DC-WEST-01'):")
print(json.dumps(result, indent=2))
print(f"\n  -> Shortfall: {1200 - result['available']} units (need 1200, have {result['available']})")

Registered tools: 9

Tool catalog:
  get_order(order_id: str)
    Look up order details by order ID.
    depends on: (none)

  get_shipment_status(order_id: str)
    Get the current shipment status for an order.
    depends on: {'get_order'}

  get_inventory(sku: str, dc_id: str)
    Check on-hand, reserved, and available inventory for a SKU at a specific DC.
    depends on: {'get_order'}

  find_alternate_inventory(sku: str, region: str)
    Search for available inventory of a SKU across DCs in a region. Use region='ALL' to search everywhere.
    depends on: {'get_inventory'}

  get_transfer_eta(from_dc: str, to_dc: str, sku: str, qty: int)
    Estimate transfer time and cost to move units between DCs.
    depends on: {'find_alternate_inventory'}

  get_supplier_expedite_options(sku: str, qty: int)
    Get available supplier expedite (rush) options for a SKU and quantity.
    depends on: {'get_order'}

  get_fulfillment_capacity(dc_id: str, date: str)
    Check available fulfillment c

---
## 6. Structured Tool-Call Schema

The model must emit tool calls in a canonical **Nemotron-style JSON format**.
This section defines the schema, shows validation in action, and demonstrates
how invalid outputs are caught before any tool executes.

### Canonical format

```json
{
  "thought": "optional short reasoning summary",
  "tool_call": {
    "name": "get_inventory",
    "arguments": {
      "sku": "SKU-4090",
      "dc_id": "DC-WEST-01"
    }
  }
}
```

### Final answer format (terminates the loop)

```json
{
  "thought": "reasoning about the recommendation",
  "final_answer": {
    "action": "transfer from DC-EAST-02",
    "rationale": "...",
    "expected_delivery": "2026-04-18",
    "meets_committed_date": true,
    "confidence": 0.85
  }
}
```

### Validation checks

1. Top-level keys must be a subset of `{thought, tool_call, final_answer}`
2. Exactly one of `tool_call` or `final_answer` must be present
3. `tool_call.name` must be a registered tool
4. `tool_call.arguments` must match the tool's expected parameters
5. No extra or unknown argument keys

In [7]:
from src.runtime.schemas import (
    validate_tool_call, extract_json,
    ParsedToolCall, ParsedFinalAnswer, ValidationError,
    REQUIRED_KEYS, OPTIONAL_KEYS, ALLOWED_TOP_KEYS,
)
from src.envs.validators import check_dependencies
from src.runtime.tools import TOOL_REGISTRY, TOOL_DEPENDENCIES

print("Schema constants:")
print(f"  Required top-level keys: {REQUIRED_KEYS}")
print(f"  Optional top-level keys: {OPTIONAL_KEYS}")
print(f"  Allowed top-level keys:  {ALLOWED_TOP_KEYS}")
print(f"  Registered tools:        {sorted(TOOL_REGISTRY.keys())}")

Schema constants:
  Required top-level keys: {'tool_call'}
  Optional top-level keys: {'thought'}
  Allowed top-level keys:  {'tool_call', 'thought', 'final_answer'}
  Registered tools:        ['find_alternate_inventory', 'get_fulfillment_capacity', 'get_inventory', 'get_order', 'get_shipment_status', 'get_supplier_expedite_options', 'get_transfer_eta', 'recommend_action', 'score_recovery_options']


In [8]:
# --- Validation examples: valid tool calls ---

print("=== Valid examples ===\n")

# 1. Minimal valid tool call
raw1 = '{"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482"}}}'
r1 = validate_tool_call(raw1, TOOL_REGISTRY)
print(f"1. Minimal tool call:  {type(r1).__name__} -> {r1.tool_name}({r1.arguments})")

# 2. With thought field
raw2 = '{"thought": "Start by checking the order", "tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482"}}}'
r2 = validate_tool_call(raw2, TOOL_REGISTRY)
print(f"2. With thought:       {type(r2).__name__} -> thought='{r2.thought}'")

# 3. JSON embedded in surrounding text (common model output)
raw3 = 'Let me check the inventory.\n{"tool_call": {"name": "get_inventory", "arguments": {"sku": "SKU-4090", "dc_id": "DC-WEST-01"}}}\nDone.'
r3 = validate_tool_call(raw3, TOOL_REGISTRY)
print(f"3. Extracted from text: {type(r3).__name__} -> {r3.tool_name}")

# 4. Final answer
raw4 = '{"thought": "Transfer is the best option", "final_answer": {"action": "transfer", "rationale": "lowest cost"}}'
r4 = validate_tool_call(raw4, TOOL_REGISTRY)
print(f"4. Final answer:       {type(r4).__name__} -> {r4.answer}")

=== Valid examples ===

1. Minimal tool call:  ParsedToolCall -> get_order({'order_id': 'SO-10482'})
2. With thought:       ParsedToolCall -> thought='Start by checking the order'
3. Extracted from text: ParsedToolCall -> get_inventory
4. Final answer:       ParsedFinalAnswer -> {'action': 'transfer', 'rationale': 'lowest cost', 'expected_delivery': None, 'meets_committed_date': None, 'confidence': None}


In [9]:
# --- Validation examples: invalid tool calls ---

print("=== Invalid examples ===\n")

# 1. No JSON at all
raw_bad1 = "I think we should check the order first."
r = validate_tool_call(raw_bad1, TOOL_REGISTRY)
print(f"1. No JSON:           {r.error_type}: {r.message}")

# 2. Missing tool_call key (flat structure)
raw_bad2 = '{"name": "get_order", "arguments": {"order_id": "SO-10482"}}'
r = validate_tool_call(raw_bad2, TOOL_REGISTRY)
print(f"2. Missing tool_call: {r.error_type}: {r.message}")

# 3. Unknown tool name
raw_bad3 = '{"tool_call": {"name": "query_database", "arguments": {"sql": "SELECT *"}}}'
r = validate_tool_call(raw_bad3, TOOL_REGISTRY)
print(f"3. Unknown tool:      {r.error_type}: {r.message[:80]}...")

# 4. Missing required arguments
raw_bad4 = '{"tool_call": {"name": "get_inventory", "arguments": {"sku": "SKU-4090"}}}'
r = validate_tool_call(raw_bad4, TOOL_REGISTRY)
print(f"4. Missing args:      {r.error_type}: {r.message}")

# 5. Extra arguments
raw_bad5 = '{"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482", "verbose": true}}}'
r = validate_tool_call(raw_bad5, TOOL_REGISTRY)
print(f"5. Extra args:        {r.error_type}: {r.message}")

=== Invalid examples ===

1. No JSON:           no_json: No JSON object found in model output.
2. Missing tool_call: extra_keys: Unexpected top-level keys: {'name', 'arguments'}. Allowed: {'tool_call', 'thought', 'final_answer'}.
3. Unknown tool:      unknown_tool: Unknown tool 'query_database'. Known tools: ['find_alternate_inventory', 'get_fu...
4. Missing args:      missing_arguments: Missing arguments for 'get_inventory': {'dc_id'}. Expected: {'sku', 'dc_id'}.
5. Extra args:        extra_arguments: Unexpected arguments for 'get_order': {'verbose'}. Expected: {'order_id'}.


In [10]:
# --- Dependency checking ---

print("=== Sequence dependency checks ===\n")

# Valid: get_order has no prerequisites
ok, reason = check_dependencies("get_order", [], TOOL_DEPENDENCIES)
print(f"get_order (no prior calls):           valid={ok}")

# Valid: get_shipment_status after get_order
ok, reason = check_dependencies("get_shipment_status", ["get_order"], TOOL_DEPENDENCIES)
print(f"get_shipment_status after get_order:  valid={ok}")

# Invalid: get_transfer_eta before find_alternate_inventory
ok, reason = check_dependencies("get_transfer_eta", ["get_order", "get_inventory"], TOOL_DEPENDENCIES)
print(f"get_transfer_eta (missing prereq):    valid={ok}")
print(f"  reason: {reason}")

# Invalid: recommend_action before score_recovery_options
ok, reason = check_dependencies("recommend_action", ["get_order", "get_inventory"], TOOL_DEPENDENCIES)
print(f"recommend_action (missing prereq):    valid={ok}")
print(f"  reason: {reason}")

=== Sequence dependency checks ===

get_order (no prior calls):           valid=True
get_shipment_status after get_order:  valid=True
get_transfer_eta (missing prereq):    valid=False
  reason: Cannot call 'get_transfer_eta' yet. Missing prerequisites: {'find_alternate_inventory'}. Tools called so far: ['get_order', 'get_inventory'].
recommend_action (missing prereq):    valid=False
  reason: Cannot call 'recommend_action' yet. Missing prerequisites: {'score_recovery_options'}. Tools called so far: ['get_order', 'get_inventory'].


---
## 7. Agent Loop

The agent loop uses the **NAT (NeMo Agent Toolkit)** runtime with an
**OpenCode-inspired** think → act → observe cycle:

```
think -> emit tool call -> validate -> execute -> observe -> continue
```

Tools are registered in a NAT `FunctionGroup` and model calls go through a
`NIMModelConfig`, so the notebook exercises the same dispatch surface that
NeMo Gym rollouts use. The core parse → validate → fallback → observe
logic is shared via `_episode_loop_core()` to keep the loop visible and
easy to follow.

Key design choices:

- **NAT tool dispatch**: tools are registered in a `FunctionGroup` via
  `build_nat_function_group()` and executed through `invoke_tool_via_group()`.
- **NIM model config**: model calls use `call_model_nim()` with a
  `NIMModelConfig`, matching the NAT LLM provider contract.
- **Bounded iterations**: hard stop at 15 iterations to prevent runaway loops.
- **Validation before execution**: every tool call is checked against both the
  schema *and* the dependency graph *before* the tool runs.
- **Error feedback**: when the model produces invalid output, the error is fed
  back as a user message so the model can self-correct on the next turn.
- **Full trace capture**: every step is recorded in an `AgentTrace` so the
  notebook can replay and evaluate the trajectory afterward.

### Architecture

```
                     +------------------+
                     |  System prompt   |
                     | (tools, format)  |
                     +--------+---------+
                              |
                     +--------v---------+
              +----->| call_model_nim() |  <- NIMModelConfig adapter
              |      +--------+---------+
              |               |
              |      +--------v---------+
              |      | validate_tool_   |  <- schema + dependency check
              |      | call()           |
              |      +--------+---------+
              |               |
              |      +--------v---------+
              |      | FunctionGroup    |  <- NAT tool dispatch
              |      | execute          |
              |      +--------+---------+
              |               |
              |      +--------v---------+
              +------+ append to trace  |  <- full trajectory captured
                     +------------------+
```

### Stop conditions

The loop terminates when:
1. The model emits a **final answer** (`final_answer` key instead of `tool_call`).
2. **Max iterations** reached (safety bound).
3. An **unrecoverable error** occurs (e.g. model endpoint down).

In [11]:
from src.runtime.agent import (
    MODEL_ENDPOINT, MODEL_NAME, MAX_ITERATIONS,
    AgentTrace, ToolCallRecord,
    run_agent_nat, print_trace_summary, trace_to_trajectory,
)
from src.runtime.nat_llm import build_nim_config
from src.runtime.prompts import build_system_prompt, build_task_message

nim_config = build_nim_config()

print("=== Agent loop configuration (NAT) ===\n")
print(f"  Model endpoint:  {MODEL_ENDPOINT}")
print(f"  Model name:      {nim_config.model_name}")
print(f"  Max iterations:  {MAX_ITERATIONS}")
print(f"  Dispatch:        NAT FunctionGroup + NIMModelConfig")

# Show the system prompt (truncated)
prompt = build_system_prompt()
print(f"\n=== System prompt ({len(prompt)} chars) ===\n")
print(prompt[:500])
print("...")

=== Agent loop configuration (NAT) ===

  Model endpoint:  http://172.17.0.1:8000/v1/chat/completions
  Model name:      nvidia/nemotron-3-nano
  Max iterations:  15
  Dispatch:        NAT FunctionGroup + NIMModelConfig

=== System prompt (2754 chars) ===

You are a supply-chain recovery agent. Your job is to diagnose order risks and recommend mitigation actions.

You have access to the following tools:
  - find_alternate_inventory(sku: str, region: str): Search for available inventory of a SKU across DCs in a region. Use region='ALL' to search everywhere. [requires: get_inventory]
  - get_fulfillment_capacity(dc_id: str, date: str): Check available fulfillment capacity at a DC on a specific date. [requires: get_order]
  - get_inventory(sku: str, 
...


### Live agent run (NAT)

The cell below runs the NAT-backed agent loop against the real Nemotron model
to investigate order SO-10482. Tools are dispatched through a NAT
`FunctionGroup` and model calls use `NIMModelConfig`.

The agent will:
1. Receive the task via the system prompt and user message
2. Emit structured tool calls one at a time
3. Have each call validated against the schema and dependency graph
4. Receive tool results as observations
5. Continue until it emits a final answer or hits the iteration limit

**Note**: The model may produce some validation errors along the way.
This is expected with smaller models -- the error feedback loop lets the
model self-correct. Section 8 adds more robust fallback parsing to reduce
these errors further.

In [12]:
# Run the NAT-backed agent loop against the live model
trace = run_agent_nat("SO-10482", verbose=True, temperature=0.1)

Using provided input_schema for multi-argument function
Using provided input_schema for multi-argument function
Using provided input_schema for multi-argument function
Using provided input_schema for multi-argument function
Using provided input_schema for multi-argument function
Using provided input_schema for multi-argument function
Using provided input_schema for multi-argument function
Using provided input_schema for multi-argument function
Using provided input_schema for multi-argument function


    Model: nvidia/nemotron-3-nano via NIMModelConfig
=== NAT agent episode started for SO-10482 ===
    Max iterations: 15

--- Iteration 1 ---
  Model: 
{
  "thought": "Start by retrieving the order details for SO-10482 to understand its SKU, committed date, source DC, and other relevant information.",
  "tool_call": {
    "name": "get_order",
    "a...
  -> Calling: get_order({'order_id': 'SO-10482'})
  -> Result: {
  "order_id": "SO-10482",
  "customer": "Acme Manufacturing",
  "sku": "SKU-4090",
  "qty": 1200,
  "source_dc": "DC-WEST-01",
  "committed_date": "2026-04-18",
  "priority": "high",
  "region": "WEST"
}
--- Iteration 2 ---
  Model: 
{
  "thought": "Retrieve the current shipment status for order SO-10482 to see if it has shipped and any delays.",
  "tool_call": {
    "name": "get_shipment_status",
    "arguments": {
      "order_...
  -> Calling: get_shipment_status({'order_id': 'SO-10482'})
  -> Result: {
  "order_id": "SO-10482",
  "status": "pending",
  "shipped_qty": 

In [13]:
# Compact trace summary
print_trace_summary(trace)

Task: Investigate order SO-10482
Status: completed (final_answer)
Tool calls: 8 valid, 0 errors
Fallbacks: 0 repairs, 0 rejects
Wall time: 37.15s (9 model calls)

Tool call sequence:
   1. [OK] get_order({'order_id': 'SO-10482'}) "Start by retrieving the order details for SO-10482 to understand its SKU, committed date, source DC, and other relevant information."
   2. [OK] get_shipment_status({'order_id': 'SO-10482'}) "Retrieve the current shipment status for order SO-10482 to see if it has shipped and any delays."
   3. [OK] get_inventory({'sku': 'SKU-4090', 'dc_id': 'DC-WEST-01'}) "Check current on-hand, reserved, and available inventory for SKU-4090 at the source DC (DC-WEST-01) to determine if we have enough stock to fulfill the order."
   4. [OK] find_alternate_inventory({'sku': 'SKU-4090', 'region': 'ALL'}) "Search for available inventory of SKU-4090 across all distribution centers to locate additional stock that can fulfill the remaining quantity needed (900 units)."
   5. [OK] 

In [14]:
# Export the trace as a trajectory (preview for the training export discussion in Section 11)
trajectory = trace_to_trajectory(trace)
print(f"Trajectory: {len(trajectory)} steps\n")
for step in trajectory:
    status = "VALID" if step["valid"] else "INVALID"
    if step["tool_name"] == "<final_answer>":
        print(f"  [{status}] final_answer:")
        for k, v in step["result"].items():
            print(f"           {k}: {v}")
    else:
        print(f"  [{status}] {step['tool_name']}({step['arguments']})")
        if not step["valid"]:
            print(f"           error: {step['validation_error']}")

Trajectory: 9 steps

  [VALID] get_order({'order_id': 'SO-10482'})
  [VALID] get_shipment_status({'order_id': 'SO-10482'})
  [VALID] get_inventory({'sku': 'SKU-4090', 'dc_id': 'DC-WEST-01'})
  [VALID] find_alternate_inventory({'sku': 'SKU-4090', 'region': 'ALL'})
  [VALID] get_fulfillment_capacity({'dc_id': 'DC-WEST-01', 'date': '2026-04-18'})
  [VALID] get_transfer_eta({'from_dc': 'DC-EAST-02', 'to_dc': 'DC-WEST-01', 'sku': 'SKU-4090', 'qty': 900})
  [VALID] get_supplier_expedite_options({'sku': 'SKU-4090', 'qty': 900})
  [VALID] score_recovery_options({'options': [{'name': 'Transfer from DC-EAST-02 to DC-WEST-01', 'lead_days': 4, 'cost_total': 2250.0, 'feasible': True, 'description': 'Move 900 units from DC-EAST-02 to DC-WEST-01'}, {'name': 'Supplier expedite - GlobalChip Express', 'lead_days': 7, 'cost_total': 7200.0, 'feasible': True, 'description': 'Rush order from GlobalChip Express'}, {'name': 'Supplier expedite - FastSemi Direct', 'lead_days': 5, 'cost_total': 10800.0, 'feasibl

### Comparison: local Python loop (without NAT)

The NAT agent loop above dispatches tools through a `FunctionGroup` and calls
the model via `NIMModelConfig` — matching the runtime surface that NeMo Gym
rollouts use. But you can also build the same think → act → observe cycle
with a plain Python loop that calls the model over raw HTTP and dispatches
tools through a dictionary.

The cell below runs the **local Python loop** (`run_agent`) against the same
order. It produces an identical `AgentTrace`, but bypasses NAT entirely:

| | NAT loop (`run_agent_nat`) | Local loop (`run_agent`) |
|---|---|---|
| **Tool dispatch** | NAT `FunctionGroup` | Plain `TOOL_REGISTRY` dict |
| **Model calls** | `call_model_nim()` via `NIMModelConfig` | `call_model()` via raw `requests.post` |
| **Core cycle** | Shared `_episode_loop_core()` | Shared `_episode_loop_core()` |
| **Output** | `AgentTrace` | `AgentTrace` |

The local loop is useful for prototyping and debugging when you want zero
framework dependencies. Once the agent behavior is correct, switching to the
NAT loop is a one-line change and gives you `FunctionGroup` registration,
NIM config management, and compatibility with NAT inspection tooling.

In [15]:
from src.runtime.agent import run_agent

# Same order, same model, same shared loop core — but no NAT layer
trace_local = run_agent("SO-10482", verbose=True, temperature=0.1)

print()
print_trace_summary(trace_local)

=== Agent episode started for SO-10482 ===
    Max iterations: 15

--- Iteration 1 ---
  Model: 
{
  "thought": "Start by retrieving the order details for SO-10482 to understand its SKU, committed date, source DC, and other relevant information.",
  "tool_call": {
    "name": "get_order",
    "a...
  -> Calling: get_order({'order_id': 'SO-10482'})
  -> Result: {
  "order_id": "SO-10482",
  "customer": "Acme Manufacturing",
  "sku": "SKU-4090",
  "qty": 1200,
  "source_dc": "DC-WEST-01",
  "committed_date": "2026-04-18",
  "priority": "high",
  "region": "WEST"
}
--- Iteration 2 ---
  Model: 
{
  "thought": "Next, retrieve the current shipment status for order SO-10482 to see if it has shipped and any delays.",
  "tool_call": {
    "name": "get_shipment_status",
    "arguments": {
      "...
  -> Calling: get_shipment_status({'order_id': 'SO-10482'})
  -> Result: {
  "order_id": "SO-10482",
  "status": "pending",
  "shipped_qty": 0,
  "carrier": null,
  "eta": null,
  "last_update": "20

---
## 8. Fallback Parsing

Real models produce messy outputs. The fallback layer sits between the raw
model output and the schema validator, attempting **mechanical repairs** before
validation. If the output cannot be salvaged, it is **rejected** with a clear
reason.

### Repair-vs-reject policy

| Failure mode | Policy | Rationale |
|---|---|---|
| Trailing commas in JSON | **Repair** | Mechanical fix, intent is clear |
| Single quotes instead of double | **Repair** | Common model error, unambiguous |
| Missing closing braces | **Repair** | Truncated output, can count depth |
| Markdown code fences around JSON | **Repair** | Strip wrapper, extract content |
| Text before/after JSON block | **Repair** | Extract first balanced `{...}` |
| Flat tool call (missing wrapper) | **Repair** | Wrap `{name, arguments}` in `tool_call` |
| Close-match tool name (edit dist <= 2) | **Repair** | e.g., `get_ordr` -> `get_order` |
| Extra unexpected arguments | **Repair** | Remove extras, keep required |
| No JSON object at all | **Reject** | Cannot determine intent |
| Missing `tool_call` and `final_answer` | **Reject** | Ambiguous structure |
| Unknown tool name (no close match) | **Reject** | Could mean anything |
| Missing required arguments | **Reject** | Cannot safely guess values |
| Unsafe argument values (SQL, shell) | **Reject** | Security boundary |

Each repair is tagged with a short label (e.g., `fixed_trailing_commas`,
`corrected_tool_name:get_ordr->get_order`) so the full repair chain is
inspectable in traces.

The `FallbackResult` dataclass records:
- `action`: `REPAIRED`, `REJECTED`, or `NO_ACTION`
- `original`: the raw model output
- `repaired`: the fixed JSON (if repaired)
- `rejection_reason`: why it was rejected (if rejected)
- `repairs_applied`: list of repair labels applied

In [ ]:
from src.runtime.fallbacks import try_repair, parse_with_fallback, FallbackAction, FallbackResult
from src.runtime.tools import TOOL_REGISTRY

# Helper to display results concisely
def show_fallback(label: str, raw: str, result: FallbackResult) -> None:
    print(f"--- {label} ---")
    print(f"  Input:   {raw[:80]}{'...' if len(raw) > 80 else ''}")
    print(f"  Action:  {result.action.value}")
    if result.repairs_applied:
        print(f"  Repairs: {result.repairs_applied}")
    if result.repaired:
        print(f"  Output:  {result.repaired[:80]}{'...' if len(result.repaired) > 80 else ''}")
    if result.rejection_reason:
        print(f"  Reason:  {result.rejection_reason}")
    print()

### 8a. Repair examples

These outputs are malformed but the intent is clear. The fallback layer
applies mechanical fixes and produces valid tool calls.

In [ ]:
# --- Repair: trailing commas ---
raw_trailing = '{"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482",},}}'
show_fallback("Trailing commas", raw_trailing, try_repair(raw_trailing, TOOL_REGISTRY))

# --- Repair: mixed text + JSON ---
raw_mixed = 'Let me check the order details first.\n{"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482"}}}\nI will analyze the result next.'
show_fallback("Mixed text + JSON", raw_mixed, try_repair(raw_mixed, TOOL_REGISTRY))

# --- Repair: single quotes ---
raw_single = "{'tool_call': {'name': 'get_order', 'arguments': {'order_id': 'SO-10482'}}}"
show_fallback("Single quotes", raw_single, try_repair(raw_single, TOOL_REGISTRY))

# --- Repair: markdown code fence ---
raw_fenced = '```json\n{"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482"}}}\n```'
show_fallback("Markdown fence", raw_fenced, try_repair(raw_fenced, TOOL_REGISTRY))

# --- Repair: truncated JSON (missing closing braces) ---
raw_truncated = '{"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482"}'
show_fallback("Truncated JSON", raw_truncated, try_repair(raw_truncated, TOOL_REGISTRY))

# --- Repair: close-match tool name ---
raw_fuzzy = '{"tool_call": {"name": "get_ordr", "arguments": {"order_id": "SO-10482"}}}'
show_fallback("Fuzzy tool name", raw_fuzzy, try_repair(raw_fuzzy, TOOL_REGISTRY))

# --- Repair: flat tool call (missing tool_call wrapper) ---
raw_flat = '{"name": "get_order", "arguments": {"order_id": "SO-10482"}}'
show_fallback("Flat tool call", raw_flat, try_repair(raw_flat, TOOL_REGISTRY))

### 8b. Reject examples

These outputs cannot be salvaged because the intent is ambiguous or the
error is structural. The fallback layer rejects them with a clear reason.

In [ ]:
# --- Reject: no JSON at all ---
raw_no_json = 'I will now check the order status for customer SO-10482.'
show_fallback("No JSON", raw_no_json, try_repair(raw_no_json, TOOL_REGISTRY))

# --- Reject: unknown tool name, no close match ---
raw_unknown = '{"tool_call": {"name": "query_database", "arguments": {"sql": "SELECT *"}}}'
show_fallback("Unknown tool", raw_unknown, try_repair(raw_unknown, TOOL_REGISTRY))

# --- Reject: missing required arguments ---
raw_missing = '{"tool_call": {"name": "get_inventory", "arguments": {"sku": "SKU-4090"}}}'
show_fallback("Missing args", raw_missing, try_repair(raw_missing, TOOL_REGISTRY))

# --- Reject: unsafe argument values ---
raw_unsafe = '{"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482; DROP TABLE orders"}}}'
show_fallback("Unsafe args", raw_unsafe, try_repair(raw_unsafe, TOOL_REGISTRY))

# --- Reject: no tool_call or final_answer ---
raw_no_tc = '{"reasoning": "I need to check the order", "next_step": "lookup"}'
show_fallback("No tool_call", raw_no_tc, try_repair(raw_no_tc, TOOL_REGISTRY))

### 8c. Fallback integration in the agent loop

The fallback layer is integrated into the agent loop in `src/runtime/agent.py`.
When `validate_tool_call` returns a `ValidationError`, the loop calls
`try_repair` to attempt a mechanical fix. If the repair succeeds,
the repaired output is re-validated and execution continues normally.
If the repair fails, the error is fed back to the model for self-correction.

Each `ToolCallRecord` in the trace now includes:
- `fallback_action`: `"repaired"`, `"rejected"`, or `None`
- `repairs_applied`: list of repair labels (e.g., `["fixed_trailing_commas", "extracted_json_from_text"]`)

The `AgentTrace` also tracks aggregate counts: `fallback_repairs` and
`fallback_rejects`.

```
Model output  -->  validate_tool_call  -->  OK?  -->  Execute tool
                         |                              ^
                         | ValidationError              |
                         v                              |
                    try_repair  -->  REPAIRED?  -->  Re-validate
                         |                              
                         | REJECTED                     
                         v                              
                  Feed error to model (self-correct)
```

### 8d. Worked repair trajectory

This simulates an agent run where the model produces malformed outputs
that the fallback layer repairs before execution. We use synthetic
"model responses" to demonstrate the repair chain without requiring
a live model connection.

In [ ]:
import json
from src.runtime.schemas import validate_tool_call, ValidationError, ParsedToolCall
from src.envs.validators import check_dependencies
from src.runtime.tools import TOOL_REGISTRY, TOOL_DEPENDENCIES
from src.runtime.fallbacks import try_repair, FallbackAction

# Simulate a sequence of malformed model outputs and show the repair chain
malformed_outputs = [
    # Step 1: Trailing comma + surrounding text
    'I need to look up the order first.\n'
    '{"thought": "Looking up order details", '
    '"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482",}}}',

    # Step 2: Single-quoted JSON
    "{'thought': 'Check shipment', "
    "'tool_call': {'name': 'get_shipment_status', 'arguments': {'order_id': 'SO-10482'}}}",

    # Step 3: Markdown fence wrapping
    '```json\n'
    '{"tool_call": {"name": "get_inventory", "arguments": {"sku": "SKU-4090", "dc_id": "DC-WEST-01"}}}\n'
    '```',

    # Step 4: Typo in tool name
    '{"tool_call": {"name": "get_fulfilment_capacity", "arguments": {"dc_id": "DC-WEST-01", "date": "2026-04-18"}}}',

    # Step 5: Truncated JSON
    '{"tool_call": {"name": "find_alternate_inventory", "arguments": {"sku": "SKU-4090", "region": "ALL"}',
]

called_tools: list[str] = []
print("=== Worked repair trajectory ===")
print()

for i, raw in enumerate(malformed_outputs, 1):
    print(f"--- Step {i} ---")
    display = raw[:70].replace('\n', '\\n')
    print(f"  Raw:     {display}{'...' if len(raw) > 70 else ''}")

    # First try normal validation
    result = validate_tool_call(raw, TOOL_REGISTRY)

    # If validation fails, try fallback repair
    if isinstance(result, ValidationError):
        fb = try_repair(raw, TOOL_REGISTRY)
        print(f"  Validation failed: {result.error_type}")
        print(f"  Fallback: {fb.action.value}")
        if fb.repairs_applied:
            print(f"  Repairs:  {fb.repairs_applied}")

        if fb.action == FallbackAction.REPAIRED and fb.repaired:
            result = validate_tool_call(fb.repaired, TOOL_REGISTRY)
            if isinstance(result, ParsedToolCall):
                # Check dependencies
                dep_ok, dep_msg = check_dependencies(result.tool_name, called_tools, TOOL_DEPENDENCIES)
                if dep_ok:
                    fn, _, _ = TOOL_REGISTRY[result.tool_name]
                    tool_result = fn(**result.arguments)
                    called_tools.append(result.tool_name)
                    status_key = next((k for k in ('status', 'available', 'remaining', 'total_available') if k in tool_result), None)
                    preview = f"{status_key}={tool_result[status_key]}" if status_key else 'OK'
                    print(f"  Executed: {result.tool_name} -> {preview}")
                else:
                    print(f"  Dep error: {dep_msg}")
            else:
                print(f"  Still invalid after repair: {result.message if hasattr(result, 'message') else result}")
        else:
            print(f"  Reject:  {fb.rejection_reason}")
    elif isinstance(result, ParsedToolCall):
        dep_ok, _ = check_dependencies(result.tool_name, called_tools, TOOL_DEPENDENCIES)
        if dep_ok:
            fn, _, _ = TOOL_REGISTRY[result.tool_name]
            tool_result = fn(**result.arguments)
            called_tools.append(result.tool_name)
            print(f"  Valid:   {result.tool_name} (no repair needed)")
    print()

print(f"Tools called: {called_tools}")
print(f"All {len(malformed_outputs)} malformed outputs were repaired and executed successfully.")

---
## 9. Worked Examples

This section presents two complete trajectories built from deterministic
tool calls (no live model needed). Both use the same `AgentTrace` structure
that the real agent loop produces, so the evaluators in section 10 can
score them identically.

- **9a. Successful trajectory** -- clean run, correct sequence, valid recommendation.
- **9b. Failure / repair trajectory** -- includes malformed outputs, fallback repairs,
  and a rejected call, showing how the agent recovers.

### 9a. Successful trajectory

A clean run where the agent follows the canonical diagnostic flow:

1. **diagnose_order_risk** -- `get_order` -> `get_shipment_status`
2. **assess_primary_fulfillment** -- `get_inventory` -> `get_fulfillment_capacity`
3. **evaluate_alternate_recovery_paths** -- `find_alternate_inventory` -> `get_transfer_eta` -> `get_supplier_expedite_options`
4. **synthesize_recommendation** -- `score_recovery_options` -> `recommend_action`

Expected: 9 tool calls, all valid, correct sequence, good recommendation.
DC-EAST-02 is the only viable transfer source (DC-CENTRAL-03 cannot
handle the 900-unit shortfall due to its 500-unit lane cap).

In [ ]:
from src.rollouts.scripted_traces import build_successful_episode, build_repair_episode
from src.rollouts.episode_runner import enrich_episode

# Build a scripted successful episode using canonical Episode types.
# No model is needed -- the scripted trace calls real tools in dependency order.
success_episode = build_successful_episode()

print(f"Episode:      {success_episode.task_id}")
print(f"Tool calls:   {success_episode.metrics.valid_tool_calls}")
print(f"Sequence:     {' -> '.join(success_episode.tool_names_called)}")
print(f"Final answer: {success_episode.final_answer.get('action', 'N/A')}")

# Enrich through the environment for per-step rewards
success_enriched = enrich_episode(success_episode)
print(f"\nTotal reward: {success_enriched.reward_summary.total_reward:+.4f}")
print(f"Avg step:     {success_enriched.reward_summary.avg_step_reward:+.4f}")

### 9b. Failure / repair trajectory

This trajectory simulates what happens when the model produces malformed
outputs. It includes:

- A **rejected** call (no JSON at all) that the agent retries after error feedback
- A **repaired** call (trailing comma fixed) that executes successfully
- A **repaired** call (typo in tool name fuzzy-matched) that executes successfully

The trajectory still completes with a valid recommendation, but the
extra steps reduce the efficiency score.

In [ ]:
# Build a scripted repair trajectory with fallback repairs and a rejection.
# Demonstrates validation errors, repairs, and rejects as structured events.
repair_episode = build_repair_episode()

print(f"Episode:      {repair_episode.task_id}")
print(f"Tool calls:   {repair_episode.metrics.valid_tool_calls}")
print(f"Repairs:      {repair_episode.metrics.repair_attempts}")
print(f"Rejects:      {repair_episode.metrics.rejects}")
print(f"Sequence:     {' -> '.join(repair_episode.tool_names_called)}")
print(f"Final answer: {repair_episode.final_answer.get('action', 'N/A')}")

# Enrich through the environment for per-step rewards
repair_enriched = enrich_episode(repair_episode)
print(f"\nTotal reward: {repair_enriched.reward_summary.total_reward:+.4f}")
print(f"Avg step:     {repair_enriched.reward_summary.avg_step_reward:+.4f}")

---
## 10. Evaluation

Trajectories are evaluated across **seven dimensions**, each producing
a normalized score from 0.0 to 1.0:

| Dimension | What it measures | Weight |
|-----------|-----------------|--------|
| Skill selection | Did the agent pick the right skills in order? | 15% |
| Tool validity | Were all tool calls well-formed JSON? | 15% |
| Tool accuracy | Were tool arguments correct for the scenario? | 10% |
| Sequence correctness | Were dependency constraints respected? | **25%** |
| Task success | Did the agent reach a valid recommendation? | 15% |
| Recovery quality | If fallback was needed, was repair correct? | 10% |
| Efficiency | Steps taken vs. optimal trajectory length | 10% |

**Sequence correctness** carries the highest weight (25%) because it is
the distinguishing evaluator for this workshop: it checks that tools
were called in an order consistent with the dependency graph, not just
that the final answer was correct.

In [ ]:
from src.eval.metrics import evaluate_trajectory
from src.eval.reports import print_evaluation

print("=" * 90)
print("EVALUATION: Successful trajectory")
print("=" * 90)
success_eval = evaluate_trajectory(success_enriched.episode)
print_evaluation(success_eval)

In [ ]:
print("=" * 90)
print("EVALUATION: Failure / repair trajectory")
print("=" * 90)
repair_eval = evaluate_trajectory(repair_enriched.episode)
print_evaluation(repair_eval)

In [ ]:
# Side-by-side comparison
from src.eval.metrics import EVAL_DIMENSIONS

print(f"{'Dimension':<25} {'Success':>8} {'Repair':>8}  {'Delta':>7}")
print("-" * 55)

s_scores = {s.dimension: s.score for s in success_eval.scores}
r_scores = {s.dimension: s.score for s in repair_eval.scores}

for dim in EVAL_DIMENSIONS:
    s = s_scores[dim]
    r = r_scores[dim]
    delta = r - s
    marker = "" if delta == 0 else ("  ^" if delta > 0 else "  v")
    print(f"{dim:<25} {s:>7.2f}  {r:>7.2f}  {delta:>+6.2f}{marker}")

print("-" * 55)
print(f"{'Overall':<25} {success_eval.overall:>7.2f}  {repair_eval.overall:>7.2f}  "
      f"{repair_eval.overall - success_eval.overall:>+6.2f}")
print()
print(f"Success trajectory: {success_eval.summary}")
print(f"Repair  trajectory: {repair_eval.summary}")

### Sequence correctness deep dive

The cell below demonstrates what happens when the dependency graph
is violated. We construct a trace with a deliberate out-of-order
call (`get_transfer_eta` before `find_alternate_inventory`) and
show how the evaluator catches it.

In [ ]:
# Build an episode with a deliberate sequence violation
from src.rollouts.trace_types import (
    Episode, EpisodeMetrics, Event, EventType,
    ToolCallPayload, ToolResultPayload, TerminalOutcomePayload,
)
from src.runtime.tools import TOOL_REGISTRY

bad_events: list[Event] = []
step = 0

# Correct start
for name, args in [
    ("get_order", {"order_id": "SO-10482"}),
    ("get_shipment_status", {"order_id": "SO-10482"}),
    ("get_inventory", {"sku": "SKU-4090", "dc_id": "DC-WEST-01"}),
]:
    fn, _, _ = TOOL_REGISTRY[name]
    result = fn(**args)
    bad_events.append(Event(event_type=EventType.TOOL_CALL, step_index=step,
                            payload=ToolCallPayload(tool_name=name, arguments=args)))
    step += 1
    bad_events.append(Event(event_type=EventType.TOOL_RESULT, step_index=step,
                            payload=ToolResultPayload(tool_name=name, result=result)))
    step += 1

# VIOLATION: get_transfer_eta BEFORE find_alternate_inventory
fn, _, _ = TOOL_REGISTRY["get_transfer_eta"]
args = {"from_dc": "DC-EAST-02", "to_dc": "DC-WEST-01", "sku": "SKU-4090", "qty": 900}
result = fn(**args)
bad_events.append(Event(event_type=EventType.TOOL_CALL, step_index=step,
                        payload=ToolCallPayload(tool_name="get_transfer_eta", arguments=args)))
step += 1
bad_events.append(Event(event_type=EventType.TOOL_RESULT, step_index=step,
                        payload=ToolResultPayload(tool_name="get_transfer_eta", result=result)))

bad_episode = Episode(
    task_id="SO-10482",
    task_prompt="Investigate order SO-10482 (bad sequence)",
    model_id="scripted",
    events=bad_events,
    metrics=EpisodeMetrics(total_steps=4, valid_tool_calls=4, model_calls=5),
)

from src.eval.metrics import eval_sequence_correctness
seq_score = eval_sequence_correctness(bad_episode)
print(f"Sequence correctness score: {seq_score.score}")
print(f"Details: {seq_score.details}")
print()
print("The evaluator caught that get_transfer_eta was called before")
print("its prerequisite find_alternate_inventory -- exactly the kind")
print("of sequence error that a final-answer-only evaluator would miss.")

---
## 11. Training-Oriented Discussion

This section connects the workshop patterns to a concrete training path.
It shows how the traces, evaluation scores, and fallback repair cases from
sections 7–10 become training data and reward signals, culminating in a
real GRPO training handoff through `NeMo RL`.

**What this section covers:**
- **11a–11c**: Supervision levels, reward decomposition, and trajectory export
- **11e**: A complete GRPO training run — rollout collection, datum group
  assembly, training step execution, ATIF trace export, and reward visualization
- **11f**: Optional NeMo Guardrails follow-up for policy enforcement

**Reference frames:**
- **`NeMo RL`** — training-oriented DatumSpec groups and GRPO backend
- **NAT ATIF** — Agent Trajectory Interchange Format for external inspection
- **Reward shaping** — decomposed dense + trajectory reward design per curriculum stage


### 11a. What should be supervised?

Two levels of supervision are possible:

| Level | Signal | Granularity | Pros | Cons |
| --- | --- | --- | --- | --- |
| **Skill** | Which skill the agent selected | 1 label per 2–3 tool calls | Coarser, more robust | Loses tool-level detail |
| **Trajectory** | Full tool call sequence | 1 label per step | Fine-grained, captures ordering | Noisier (multiple valid orderings) |

For this workshop, we supervise at **both** levels:
- Skill-level labels drive the `skill_selection` reward dimension.
- Trajectory-level labels drive `tool_validity`, `sequence_correctness`,
  `tool_accuracy`, and `efficiency`.
- `task_success` and `recovery_quality` are outcome-level signals.

### 11b. Reward decomposition

An earlier reward-shaping framing treats reward design as a composition of
interpretable components rather than a single scalar. For this agent, we
decompose the reward at two time scales:

**Per-step (dense) rewards** — applied after each tool call:

| Component | Weight | Signal |
| --- | --- | --- |
| `tool_validity` | 0.25 | +1 if valid call, −0.5 if invalid |
| `sequence_correctness` | 0.35 | +1 if deps satisfied, −1 if violated |
| `tool_accuracy` | 0.20 | fraction of expected arguments correct |
| `recovery_bonus` | 0.20 | +0.5 if fallback repair succeeded |

**Trajectory-level (sparse) reward** — applied at episode end:

The weighted evaluation score from the seven-dimension evaluator
(section 10), with `sequence_correctness` carrying the highest weight
(0.25) to avoid rewarding correct answers reached via invalid paths.

The combined reward blends step-level (40%) and trajectory-level (60%)
signals, balancing dense learning signal with outcome-oriented scoring.

In [ ]:
# Display per-step reward breakdown from the environment enrichment.
# Rewards were computed by replaying episodes through LateOrderRecoveryEnv.

def print_enriched_reward_breakdown(label, enriched_result, evaluation):
    """Print per-step rewards from an enriched episode."""
    rs = enriched_result.reward_summary
    ep = enriched_result.episode

    print(f"{label}")
    print("=" * 75)
    print(f"{'Step':>4}  {'Tool':<30}  {'Reward':>7}  Penalties")
    print("-" * 75)

    for i, sr in enumerate(rs.step_rewards):
        penalties = ", ".join(sr.penalties) if sr.penalties else ""
        # Find the tool name for this step
        tool_calls = ep.tool_names_called
        tool_name = tool_calls[i] if i < len(tool_calls) else "<unknown>"
        print(f"{i:>4}  {tool_name:<30}  {sr.total:>+7.4f}  {penalties}")

    if rs.terminal_reward:
        penalties = ", ".join(rs.terminal_reward.penalties) if rs.terminal_reward.penalties else ""
        print(f"{'T':>4}  {'<terminal>':30}  {rs.terminal_reward.total:>+7.4f}  {penalties}")

    print("-" * 75)
    print(f"Total reward:     {rs.total_reward:+.4f}")
    print(f"Avg step reward:  {rs.avg_step_reward:+.4f}")
    print(f"Eval overall:     {evaluation.overall:.3f}")
    print()


print_enriched_reward_breakdown("SUCCESSFUL TRAJECTORY", success_enriched, success_eval)
print_enriched_reward_breakdown("REPAIR TRAJECTORY", repair_enriched, repair_eval)

### 11c. Training trajectory export

The training export format uses JSONL records, where each record contains an
ordered list of `(observation, action, reward, done)` tuples. The export
function in `src/rollouts/export_adapters.py` converts an `AgentTrace` plus its
evaluation and step rewards into this format.

This is the handoff point: everything before this cell was produced by our
**OpenCode-inspired architecture**. We implement a small local version of
the loop so the trace generation logic stays easy to inspect. Everything
after would be consumed by downstream training infrastructure.

In [ ]:
from src.rollouts.export_adapters import (
    episode_to_training_trajectory,
    training_trajectory_to_jsonl,
)
import json

# Export the successful trajectory for downstream training
success_training_trajectory = episode_to_training_trajectory(
    success_enriched.episode,
    reward_summary=success_enriched.reward_summary,
)

print(f"Task:            {success_training_trajectory.task_id}")
print(f"Model:           {success_training_trajectory.model_id}")
print(f"Episode length:  {success_training_trajectory.episode_length}")
print(f"Total reward:    {success_training_trajectory.total_reward:+.4f}")
print()

# Show one JSONL record (truncated for readability)
jsonl_line = training_trajectory_to_jsonl(success_training_trajectory)
parsed = json.loads(jsonl_line)

print("JSONL record structure (first 2 steps shown):")
print(json.dumps({
    "task_id": parsed["task_id"],
    "total_reward": parsed["total_reward"],
    "episode_length": parsed["episode_length"],
    "steps": parsed["steps"][:2],
    "metadata": parsed["metadata"],
}, indent=2))

In [ ]:
# Export and compare both trajectories
repair_training_trajectory = episode_to_training_trajectory(
    repair_enriched.episode,
    reward_summary=repair_enriched.reward_summary,
)

print("Training export comparison")
print("=" * 60)
print(f"{'Metric':<30} {'Success':>12} {'Repair':>12}")
print("-" * 60)
print(f"{'Episode length':<30} {success_training_trajectory.episode_length:>12} {repair_training_trajectory.episode_length:>12}")
print(f"{'Total reward':<30} {success_training_trajectory.total_reward:>+12.4f} {repair_training_trajectory.total_reward:>+12.4f}")
print(f"{'Valid tool calls':<30} {success_training_trajectory.metadata['valid_tool_calls']:>12} {repair_training_trajectory.metadata['valid_tool_calls']:>12}")
print(f"{'Invalid tool calls':<30} {success_training_trajectory.metadata['invalid_tool_calls']:>12} {repair_training_trajectory.metadata['invalid_tool_calls']:>12}")
print(f"{'Repair attempts':<30} {success_training_trajectory.metadata['repair_attempts']:>12} {repair_training_trajectory.metadata['repair_attempts']:>12}")
print(f"{'Rejects':<30} {success_training_trajectory.metadata['rejects']:>12} {repair_training_trajectory.metadata['rejects']:>12}")
print()
print("The successful trajectory earns a higher total reward because:")
print("- No invalid steps (no negative validity penalties)")
print("- Fewer total steps (better efficiency in trajectory-level score)")
print("- The repair trajectory still passes thanks to recovery quality")

### 11e. GRPO training run

The cells below replace the earlier export-only workflow with a **real GRPO
training handoff**. The sequence is:

1. **Collect rollouts** — gather multiple enriched episodes from the scripted
   traces (a mix of successful and repair trajectories)
2. **Build GRPO group** — assemble a DatumSpec group with stage-shaped rewards
   and group-relative advantages
3. **Execute training step** — run one GRPO training step through `NeMo RL`
   (dry-run mode when no GPU backend is available)
4. **Export ATIF traces** — write NAT-compatible ATIF JSONL for external inspection
5. **Visualize rewards** — plot reward distributions, advantages, and per-step
   shaped reward heatmaps

All orchestration logic lives in `src/training/grpo_notebook.py`, keeping
the notebook as a consumer rather than a source of truth.

```
  NAT Rollouts ──▶ Enriched Episodes ──▶ GRPO DatumSpec Group
       │                    │                      │
       │                    ▼                      ▼
       │             ATIF Export             NeMo RL Train
       │                    │                      │
       │                    ▼                      ▼
       │           NAT Inspection         Checkpoints + Metrics
       │                                           │
       └─────────▶ Reward Views ──────────▶ Reward Plots
```


### Note on rollout collection

The rollout collection below uses **scripted episodes** — pre-designed action
sequences executed through the environment-backed pipeline. This ensures
reproducibility and makes the GRPO demonstration deterministic.

In a real GRPO training loop, rollouts would be collected by running
`run_agent_episode()` with the current policy model, producing diverse
model-generated trajectories. The group-relative advantage computation
would then operate on genuine policy variance rather than artificial variance.

The scripted approach exercises the same execution pipeline (validate →
repair → reject → execute → record) and produces episodes with identical
trace structure, reward annotations, and DatumSpec format as live rollouts.

In [ ]:
# Step 1: Collect enriched rollouts (mix of successful + repair trajectories)
# Step 2: Build GRPO trajectory group with group-relative advantages

from src.training.grpo_notebook import (
    collect_enriched_rollouts,
    build_grpo_group_from_rollouts,
    extract_reward_plot_data,
)
from src.training.curriculum import TrainingStage

# Collect 4 episodes: alternating success/repair for GRPO diversity
grpo_rollouts = collect_enriched_rollouts(num_rollouts=4, include_repairs=True)

print(f"Collected {len(grpo_rollouts)} enriched episodes:")
for i, r in enumerate(grpo_rollouts):
    ep = r.episode
    print(f"  ep{i}: {ep.metrics.valid_tool_calls} valid calls, "
          f"{ep.metrics.repair_attempts} repairs, "
          f"reward={r.reward_summary.total_reward:+.4f}")

# Build GRPO group using the full_multistep_rl stage
grpo_group, grpo_stage, grpo_views = build_grpo_group_from_rollouts(
    grpo_rollouts,
    stage=TrainingStage.FULL_MULTISTEP_RL,
)

print(f"\nGRPO DatumSpec group: {len(grpo_group)} datum specs")
print(f"Stage: {grpo_stage.stage.value}")
print(f"Reward blend: step_weight={grpo_stage.step_reward_weight}, "
      f"traj_weight={grpo_stage.trajectory_reward_weight}")

# Show per-datum rewards and advantages
print(f"\n{'Datum':<6} {'Reward':>8} {'Advantage':>10} {'Group Mean':>11}")
print("-" * 38)
for i, d in enumerate(grpo_group):
    info = d.get("extra_env_info", {})
    reward = info.get("reward", 0.0)
    adv = info.get("group_advantage", 0.0)
    gmean = info.get("group_mean_reward", 0.0)
    print(f"  {i:<4} {reward:>+8.4f} {adv:>+10.4f} {gmean:>+11.4f}")


In [ ]:
# Step 3: Execute GRPO training step
#
# In dry-run mode (default), this simulates the training step and
# produces mock metrics. Set dry_run=False to attempt a real training
# step via NeMo RL (requires distributed GPU resources and a
# configured policy model).

from src.training.grpo_notebook import run_grpo_notebook

grpo_result = run_grpo_notebook(
    num_rollouts=4,
    include_repairs=True,
    stage=TrainingStage.FULL_MULTISTEP_RL,
    artifact_dir="artifacts/grpo_run",
    dry_run=True,  # Set False for real training with a configured backend
)

grpo_result.print_summary()


In [ ]:
# Step 4: ATIF trace export for NAT inspection
#
# The GRPO run produced ATIF JSONL artifacts that can be loaded into
# NAT's trace inspection UI. These contain the full conversation with
# tool calls, observations, repairs, and terminal outcomes.

import json, os

print(f"ATIF artifact: {grpo_result.atif_path}")
print(f"TrajectoryGroup: {grpo_result.group_jsonl_path}")
print()

# Preview the first ATIF trajectory
with open(grpo_result.atif_path) as f:
    first_atif = json.loads(f.readline())

print("First ATIF trajectory preview:")
print(f"  Agent: {first_atif['agent']['name']} v{first_atif['agent']['version']}")
print(f"  Steps: {len(first_atif['steps'])}")
print(f"  Task:  {first_atif.get('extra', {}).get('task_id', 'N/A')}")
print()

# Show step summary
for step in first_atif['steps'][:6]:
    source = step.get('source', '?')
    msg = (step.get('message', '') or '')[:60]
    tool_calls = step.get('tool_calls', [])
    if tool_calls:
        fn = tool_calls[0].get('function_name', '?')
        print(f"  [{source:>6}] tool_call: {fn}")
    elif msg:
        print(f"  [{source:>6}] {msg}")
print(f"  ... ({len(first_atif['steps']) - 6} more steps)")
print()
print("To inspect in NAT UI:")
print(f"  nat inspect {os.path.abspath(grpo_result.atif_path)}")


In [ ]:
# Step 5: Reward distribution visualization
%matplotlib inline

from src.training.grpo_notebook import extract_reward_plot_data
from src.training.reward_plots import (
    plot_grpo_rewards,
    plot_reward_components,
    print_grpo_summary_table,
)

plot_data = extract_reward_plot_data(grpo_result)

# Summary table
print_grpo_summary_table(plot_data)


In [ ]:
# GRPO reward analysis plots
fig = plot_grpo_rewards(plot_data)


In [ ]:
# Reward component breakdown per episode
fig2 = plot_reward_components(plot_data, grpo_result.enriched_results)


### 11f. NeMo Guardrails (optional follow-up)

**NeMo Guardrails** would extend the fallback parsing layer (section 8)
with policy enforcement. Where the current fallback layer makes
mechanical repairs (fix JSON syntax, fuzzy-match tool names), guardrails
would add semantic checks:

- **Argument policy**: reject tool calls that reference disallowed
  values beyond the current regex-based unsafe patterns (e.g., enforce
  that SKUs must match a known catalog).
- **Business rules**: enforce domain constraints like "cannot recommend
  a substitute SKU unless the customer has pre-approved it."
- **Rate limiting**: prevent the agent from calling the same tool with
  the same arguments more than once per episode.
- **Escalation triggers**: auto-escalate to a human if the agent
  exceeds a configurable error budget per episode.

This is noted as an optional follow-up, not core MVP scope. The
workshop already demonstrates the pattern (fallback parsing + repair
vs. reject policy); NeMo Guardrails would productionize and extend it.

---
## Summary

This notebook has walked through a complete agentic supply-chain
workflow on the NVIDIA stack:

1. **Scenario** — late order recovery for SO-10482 (1,200 units at risk)
2. **Data** — small synthetic tables for orders, shipments, inventory,
   transfer lanes, supplier options, and fulfillment capacity
3. **Skills** — 4 explicit skills composing 9 deterministic tools
4. **Tools** — 7–9 tools with machine-checkable dependency ordering
5. **Schema** — canonical Nemotron-style tool-call format with validation
6. **Agent loop** — OpenCode-inspired think/emit/validate/execute/observe cycle built locally for visibility and understanding
7. **Fallback parsing** — repair-vs-reject policy for malformed outputs
8. **Worked examples** — one clean trajectory, one with repairs
9. **Evaluation** — seven-dimension scoring with sequence-sensitive checks
10. **GRPO training** — real GRPO handoff with group-relative advantages,
    ATIF trace export for NAT inspection, and reward distribution
    visualization

The patterns shown here — explicit skill composition, deterministic
tools, structured output parsing, sequence-sensitive evaluation,
reward decomposition, and GRPO training handoff via NeMo RL —
are directly applicable to production agent training on the NVIDIA
stack.
